# Notebook 04 — Interpreting the Plot-Similarity Network (v2)

**Job of this notebook:** say what the narrative communities found in Notebook 03 *are*, which films hold the network together and which films sit between communities. It also produces the labels every later notebook and brick uses.

**Reading rule:** centrality is a position in the plot-similarity network, not fame or quality. A film is central when many films have similar overviews.

### What changed from v1
| # | Change | Why it matters |
|---|--------|----------------|
| 1 | Built for the **consensus communities and exact betweenness** of NB03 v2. Every film carries its **membership confidence**; every community its **robustness**. A community with robustness below 0.5 is marked *tentative*. Still runs on a v1 network (the new parts are skipped). | Interpretations rest on boundaries that do not depend on one random seed. |
| 2 | **Representative films are ranked by within-community strength** (similarity to films of the *same* community), core films first (core = ≥ 75% of a film's similarity stays inside its community). v1 used global weighted degree, which also rewards films whose neighbours are in other communities. | The films used to name a community actually belong to it. |
| 3 | **Bridges are described, not just ranked:** exact betweenness plus the **participation coefficient** (Guimerà & Amaral, 2005) and the two communities each bridge connects. v1's bridge list came from sampled betweenness (only 9 of the top 50 survived a re-sample). | "Bridge film" becomes a measured claim: *between which narratives*. |
| 4 | **New evidence for labelling:** distinctive words per community (class-based TF-IDF on the overviews), recurring character/franchise names, genre composition, a map of which communities border which. | Labels are justified by the plots themselves, not only by a few titles. |
| 5 | **Are communities just genres?** Normalised mutual information between community and genre, with a composition heatmap. | Directly answers the obvious examiner question. |
| 6 | **Labelling workflow without copy-paste:** labels carried over by NB03 are pre-filled; you edit one file (`community_labels_v2_manual.csv`), and the labelled file the bricks read is written only when every major community has a reviewed label. A labelled file left over from the old network is detected, never silently used. | No stale labels reach the bricks. |
| 7 | Single `CFG`, idempotent Drive mount, vectorised code (no per-community loops), title families computed once, word-bounded event check (v1's `TNA` matched *Vietnam*). | Reliability and speed. |

**Files the bricks read keep their v1 names and columns** (new columns are added at the end): `top_visible_central_films_per_community.csv`, `top_weighted_degree_raw.csv`, `top_weighted_degree_franchise_capped.csv`, `top_weighted_degree_visible_franchise_capped.csv`, `top_betweenness_visible_franchise_capped.csv`, `bridging_films_betweenness.csv`, `community_manual_labeling_template_labeled.csv`. New tables and figures go to `outputs/tables/interpretation/` and `outputs/figures/interpretation/`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 1 — CONFIG
# ═══════════════════════════════════════════════════════════════════════════
CFG = {
    "project_dir": "/content/drive/MyDrive/thesis_movie_project",

    # ── what counts as interpretable ─────────────────────────────────────────
    "min_community_size_for_interpretation": 100,   # "major" communities get labels (v1: 100)
    "display_min_vote_count": 50,                   # films shown as examples must be recognisable (v1: 50)
    "core_internal_share": 0.75,                    # core film: ≥ 75% of its similarity stays inside its community
    "boundary_confidence": 0.20,                    # boundary film: in its community in < 20% of single Louvain runs
    "tentative_robustness": 0.50,                   # community robustness < this = tentative community
    # a community is "franchise/name-driven" when one title family or one character name dominates it
    # (real data: Scooby-Doo, Harry Potter, Madea, Chucky, Death Race, Lupin/Leprechaun, Coco Chanel, Oscar Wilde …)
    "franchise_family_share": 0.25,                 # largest title family ≥ this share of the community's films
    "name_film_share": 0.40,                        # top character/franchise name appears in ≥ this share of its overviews

    # ── table sizes ──────────────────────────────────────────────────────────
    "top_n_global": 50,
    "top_n_bridges": 100,
    "top_n_per_community": 10,
    "top_n_per_community_display": 3,               # rows per community in top_visible_central_films_per_community
    "n_peripheral": 100,

    # ── how representative films are chosen ──────────────────────────────────
    "central_metric": "within_community_strength",  # or "weighted_degree" (v1)
    "core_first": True,                             # within a community, list core films before boundary films

    # ── distinctive words per community (needs df_network_corpus.csv for the overviews) ──
    "key_terms": True,
    "key_terms_n": 12,
    "key_terms_min_df": 5,
    "key_terms_max_df": 0.40,
    "separate_names": True,          # character names and places are reported apart from theme words
    "use_spacy_ner": True,           # spaCy PERSON / GPE tags decide what is a name (fallback: capitalisation rule)
    "name_capital_share": 0.90,
    "well_known_min_votes": 1000,    # "well-known examples" column in the labelling evidence

    # ── labels ───────────────────────────────────────────────────────────────
    "require_review_before_export": True,   # "carried – review" and "new" labels need label_reviewed = yes

    # ── display-only franchise cap (never changes the network) ───────────────
    "franchise_patterns": {
        "scooby_doo": r"scooby[\s\-]*doo|\bscoob\b", "star_wars": r"star\s+wars",
        "harry_potter": r"harry\s+potter|fantastic\s+beasts", "lord_of_the_rings": r"lord\s+of\s+the\s+rings|hobbit",
        "batman": r"\bbatman\b|dark\s+knight", "superman": r"\bsuperman\b", "spider_man": r"spider[\s\-]*man",
        "x_men": r"\bx[\s\-]*men\b|wolverine", "avengers": r"\bavengers\b",
        "fast_furious": r"fast\s+.*furious|fast\s+five|fast\s+x", "mission_impossible": r"mission[\s:]*impossible",
        "james_bond": r"james\s+bond|007", "jurassic": r"jurassic\s+park|jurassic\s+world",
        "godzilla": r"\bgodzilla\b", "king_kong": r"king\s+kong", "pokemon": r"pok[eé]mon", "naruto": r"\bnaruto\b",
        "dragon_ball": r"dragon\s+ball", "one_piece": r"one\s+piece", "detective_conan": r"detective\s+conan|case\s+closed",
        "barbie": r"\bbarbie\b", "cinderella": r"\bcinderella\b", "snow_white": r"snow\s+white",
        "alice_wonderland": r"alice\s+.*wonderland", "christmas_carol": r"christmas\s+carol",
        "sherlock_holmes": r"sherlock\s+holmes", "zorro": r"\bzorro\b",
    },
    # word-bounded (v1 matched "TNA" inside "Vietnam")
    "event_title_pattern": r"\b(?:WWE|WWF|WCW|NXT|AEW|TNA|Impact Wrestling|WrestleMania|Royal Rumble|SummerSlam|"
                           r"Survivor Series|UFC|Bellator|ONE Championship|Pride FC)\b",
    "figure_dpi": 200,
    "label_chars_in_figures": 42,    # long labels are shortened at a word boundary ("…")
}
print("CONFIG loaded.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 2 — Imports, Drive mount (idempotent), paths, figure style, helpers
# ═══════════════════════════════════════════════════════════════════════════
import json, re, hashlib, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.stats import kruskal
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 110)
try:
    display
except NameError:
    display = print

try:
    from google.colab import drive
    if Path("/content/drive/MyDrive").exists():
        print("Google Drive already mounted.")
    else:
        drive.mount("/content/drive")
except Exception as e:
    print(f"Drive mount skipped: {e}")

PROJECT_DIR = Path(CFG["project_dir"])
CORE_DIR    = PROJECT_DIR / "processed" / "core"
NETWORK_DIR = PROJECT_DIR / "processed" / "network"
TABLES_DIR  = PROJECT_DIR / "outputs" / "tables"            # v1 location: files the bricks read
FIG_DIR     = PROJECT_DIR / "outputs" / "figures"
INT_TABLES  = TABLES_DIR / "interpretation"                 # new tables
INT_FIGS    = FIG_DIR / "interpretation"                    # new figures
for d in [TABLES_DIR, FIG_DIR, INT_TABLES, INT_FIGS]:
    d.mkdir(parents=True, exist_ok=True)

NODES_FILE    = NETWORK_DIR / "semantic_network_nodes_lcc.csv"
EDGES_FILE    = NETWORK_DIR / "semantic_network_edges_mutual_knn.csv"
SUMMARY_FILE  = NETWORK_DIR / "community_summary.csv"
AUDIT_FILE    = TABLES_DIR / "network_audit.csv"
MANIFEST_FILE = NETWORK_DIR / "network_manifest.json"
CORPUS_FILE   = CORE_DIR / "df_network_corpus.csv"
DRAFT_FILE    = TABLES_DIR / "community_labels_draft_v2.csv"       # from NB03 v2 (label carry-over)
MANUAL_FILE   = TABLES_DIR / "community_labels_v2_manual.csv"      # the ONE file you edit
LABELED_FILE  = TABLES_DIR / "community_manual_labeling_template_labeled.csv"   # read by the bricks

# ── one figure style for all notebooks (validated colour-blind-safe palette) ──
INK, INK_2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
DIVERGING = "RdBu_r"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlepad": 24,
    "axes.labelsize": 10, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "text.color": INK, "font.family": "DejaVu Sans", "lines.linewidth": 2, "legend.frameon": False,
})

def subtitle(ax, text):
    # fixed offset in points (not a fraction of the axes), so tall figures do not push it into the title
    ax.annotate(text, xy=(0, 1), xycoords="axes fraction", xytext=(0, 6), textcoords="offset points",
                fontsize=9.5, color=INK_2, va="bottom", ha="left")

import textwrap
def short(text, n=None):
    return textwrap.shorten(str(text), width=n or CFG["label_chars_in_figures"], placeholder="…")

def save(fig, table, name, also_v1=None):
    """Every new figure is saved with the table that holds its numbers."""
    fig.savefig(INT_FIGS / f"{name}.png", dpi=CFG["figure_dpi"], bbox_inches="tight")
    if also_v1:
        fig.savefig(FIG_DIR / also_v1, dpi=CFG["figure_dpi"], bbox_inches="tight")
    table.to_csv(INT_TABLES / f"{name}.csv", index=False)
    plt.show()
    print(f"  ✓ figures/interpretation/{name}.png + tables/interpretation/{name}.csv"
          + (f"  ✓ figures/{also_v1}" if also_v1 else ""))

def save_v1(df, name):
    df.to_csv(TABLES_DIR / name, index=False)
    print(f"  ✓ tables/{name}")

def save_new(df, name):
    df.to_csv(INT_TABLES / name, index=False)
    print(f"  ✓ tables/interpretation/{name}")

def fingerprint(values):
    return hashlib.sha256(",".join(pd.Series(values).astype(str)).encode()).hexdigest()[:16]

def is_blank(s):
    return s.fillna("").astype(str).str.strip().isin(["", "nan", "None"])

print("PROJECT_DIR:", PROJECT_DIR)


## 1. Load the network and check it is the one Notebook 03 wrote

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 3 — Load network outputs (required files stop the notebook, optional ones are skipped)
# ═══════════════════════════════════════════════════════════════════════════
required = {"nodes": NODES_FILE, "edges": EDGES_FILE, "community_summary": SUMMARY_FILE}
missing = [k for k, p in required.items() if not p.exists()]
if missing and Path("/content/drive").exists():
    # Colab's Drive mount can show a stale view when Notebook 03 rewrote these files from another session
    # (or is still writing them). Remount once and look again before giving up.
    try:
        from google.colab import drive
        print("Network files not visible — remounting Google Drive to refresh its view …")
        drive.mount("/content/drive", force_remount=True)
        missing = [k for k, p in required.items() if not p.exists()]
    except Exception as ex:
        print("Remount failed:", ex)
if missing:
    print("Looked in:", NETWORK_DIR)
    print("Folder exists:", NETWORK_DIR.exists())
    if NETWORK_DIR.exists():
        print("It contains:", sorted(p.name for p in NETWORK_DIR.iterdir())[:30])
    raise FileNotFoundError(
        f"Missing: {missing}. If Notebook 03 is still running in another tab, wait for it to finish; "
        "otherwise use Runtime → Restart session and run this notebook again from Cell 1.")

nodes = pd.read_csv(NODES_FILE)
edges = pd.read_csv(EDGES_FILE)
community_summary = pd.read_csv(SUMMARY_FILE)
network_audit = pd.read_csv(AUDIT_FILE) if AUDIT_FILE.exists() else pd.DataFrame(columns=["metric", "value"])
manifest = json.loads(MANIFEST_FILE.read_text()) if MANIFEST_FILE.exists() else None
print(f"nodes {nodes.shape} · edges {edges.shape} · communities {nodes['community_id'].nunique()}")

# ── which network is this? ───────────────────────────────────────────────────
IS_V2 = {"membership_confidence", "betweenness_centrality"} <= set(nodes.columns)
BETW = "betweenness_centrality" if "betweenness_centrality" in nodes else "betweenness_centrality_sampled"
HAS_CONF = IS_V2 and nodes["membership_confidence"].notna().any()
audit = network_audit.set_index("metric")["value"] if len(network_audit) else pd.Series(dtype=object)
print("Network version:", "v2 (consensus communities, exact betweenness)" if IS_V2
      else "v1 (single Louvain, sampled betweenness) — confidence-based parts are skipped")
print("Community method:", audit.get("community_method", "unknown"), "· betweenness:", audit.get("betweenness_method", BETW))
if manifest:
    ok = manifest.get("nodes_fingerprint") == fingerprint(nodes["tmdb_id"])
    print("Matches network_manifest.json:", ok)
    if not ok:
        print("⚠ The node file changed after Notebook 03 wrote its manifest — re-run Notebook 03 v2.")

# ── one id → community lookup, used everywhere below ─────────────────────────
nodes["community_id"] = nodes["community_id"].astype(int)
comm_of = nodes.set_index("node_id")["community_id"]
sizes = nodes["community_id"].value_counts()
MAJOR = sizes[sizes >= CFG["min_community_size_for_interpretation"]].index.tolist()   # largest first
PARTITION_FP = fingerprint(nodes.sort_values("tmdb_id")[["tmdb_id", "community_id"]].astype(str).agg(":".join, axis=1))
print(f"Major communities (≥ {CFG['min_community_size_for_interpretation']} films): {len(MAJOR)} "
      f"covering {sizes[MAJOR].sum() / len(nodes):.1%} of films · partition fingerprint {PARTITION_FP}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 4 — Basic checks and event contamination (same files as v1)
# ═══════════════════════════════════════════════════════════════════════════
basic_checks = pd.DataFrame([
    {"check": "n_nodes", "value": len(nodes)},
    {"check": "n_edges", "value": len(edges)},
    {"check": "n_communities", "value": nodes["community_id"].nunique()},
    {"check": "n_major_communities", "value": len(MAJOR)},
    {"check": "min_year", "value": nodes["release_year"].min()},
    {"check": "max_year", "value": nodes["release_year"].max()},
    {"check": "median_vote_count", "value": nodes["vote_count"].median()},
    {"check": "median_overview_word_count", "value": nodes["overview_word_count"].median()},
    {"check": "revenue_available_pct", "value": round(nodes["revenue_clean"].notna().mean() * 100, 2)},
    {"check": "edges_with_unknown_node", "value": int((~edges["source_node_id"].isin(comm_of.index)
                                                       | ~edges["target_node_id"].isin(comm_of.index)).sum())},
    {"check": "duplicate_tmdb_id", "value": int(nodes["tmdb_id"].duplicated().sum())},
])
if HAS_CONF:
    basic_checks.loc[len(basic_checks)] = ["median_membership_confidence", nodes["membership_confidence"].median()]
display(basic_checks)
save_v1(basic_checks, "network_interpretation_basic_checks.csv")

event_check = nodes[nodes["title"].fillna("").str.contains(CFG["event_title_pattern"], case=False, regex=True)]
print(f"Titles with a wrestling/MMA brand left in the network: {len(event_check)}")
if len(event_check):
    print("  These are story films kept on purpose by Notebook 01 (animated/family crossovers such as "
          "'Scooby-Doo! WrestleMania Mystery'), not recorded events. Check the list once.")
display(event_check[["title", "release_year", "primary_genre", "vote_count", "community_id"]].head(20))
save_v1(event_check, "remaining_event_like_titles_check.csv")


## 2. Where each film sits: within-community strength and participation

For every film, its edge weights (cosine similarities) are split by the community of each neighbour:
* **within-community strength** — total similarity to films of its own community; the basis for choosing representative films;
* **participation coefficient** *P* = 1 − Σ<sub>s</sub>(k<sub>is</sub>/k<sub>i</sub>)² — 0 when all neighbours are in one community, higher when they are spread over several (Guimerà & Amaral, 2005);
* **main external community** — where most of its outside similarity goes.

For each community: the share of its members' similarity that stays inside (**cohesion**) and mean similarity of internal vs boundary edges.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 5 — Film-level position and community cohesion (vectorised over the edge list)
# ═══════════════════════════════════════════════════════════════════════════
e = edges[["source_node_id", "target_node_id", "cosine_similarity"]].rename(
    columns={"source_node_id": "a", "target_node_id": "b", "cosine_similarity": "w"})
e["ca"], e["cb"] = e["a"].map(comm_of), e["b"].map(comm_of)
e = e.dropna(subset=["ca", "cb"]).astype({"ca": int, "cb": int})
e["internal"] = e["ca"] == e["cb"]

# both directions: every film sees each neighbour's community
both = pd.concat([e[["a", "cb", "w"]].set_axis(["node_id", "nbr_comm", "w"], axis=1),
                  e[["b", "ca", "w"]].set_axis(["node_id", "nbr_comm", "w"], axis=1)], ignore_index=True)
k_is = both.groupby(["node_id", "nbr_comm"])["w"].sum().reset_index()
k_is["own"] = k_is["nbr_comm"].values == k_is["node_id"].map(comm_of).values
k_i = k_is.groupby("node_id")["w"].sum()
k_is["share"] = k_is["w"] / k_is["node_id"].map(k_i)

position = pd.DataFrame({
    "within_community_strength": k_is[k_is["own"]].groupby("node_id")["w"].sum(),
    "participation_coefficient": 1 - (k_is["share"] ** 2).groupby(k_is["node_id"]).sum(),
})
ext = k_is[~k_is["own"]].sort_values(["node_id", "w"], ascending=[True, False]).drop_duplicates("node_id")
position["main_external_community"] = ext.set_index("node_id")["nbr_comm"]
position["external_share"] = 1 - position["within_community_strength"].fillna(0) / k_i
nodes = nodes.drop(columns=[c for c in position.columns if c in nodes.columns]).merge(
    position, left_on="node_id", right_index=True, how="left")
nodes["within_community_strength"] = nodes["within_community_strength"].fillna(0)
nodes["main_external_community"] = nodes["main_external_community"].astype("Int64")
# core film: most of its plot similarity stays inside its own community (structural, per film).
# Membership confidence is NOT used for this: it measures whether single Louvain runs keep the film's whole
# community together, so films of a community that single runs split in two all get the same value.
nodes["is_core"] = (1 - nodes["external_share"]) >= CFG["core_internal_share"]
nodes["is_boundary"] = (nodes["membership_confidence"] < CFG["boundary_confidence"]) if HAS_CONF else False

# ── community cohesion ───────────────────────────────────────────────────────
strength_by_comm = nodes.groupby("community_id")[["within_community_strength"]].sum()
strength_by_comm["total_strength"] = nodes.assign(k=nodes["node_id"].map(k_i)).groupby("community_id")["k"].sum()
w_int = e[e["internal"]].groupby("ca")["w"].mean()
w_bnd = pd.concat([e.loc[~e["internal"], ["ca", "w"]].rename(columns={"ca": "c"}),
                   e.loc[~e["internal"], ["cb", "w"]].rename(columns={"cb": "c"})]).groupby("c")["w"].mean()
cohesion = pd.DataFrame({
    "cohesion_internal_share": strength_by_comm["within_community_strength"] / strength_by_comm["total_strength"],
    "mean_internal_similarity": w_int, "mean_boundary_similarity": w_bnd,
}).rename_axis("community_id").reset_index()
cohesion = cohesion.merge(nodes.groupby("community_id").agg(core_share=("is_core", "mean")).reset_index(),
                          on="community_id")
if HAS_CONF:
    cohesion = cohesion.merge(nodes.groupby("community_id").agg(
        community_robustness=("membership_confidence", "mean"),
        boundary_share=("is_boundary", "mean")).reset_index(), on="community_id")
POSITIONS_FILE = NETWORK_DIR / "node_positions_nb04.csv"     # for Notebook 05 (merge on tmdb_id)
nodes["community_group"] = np.where(nodes["community_id"].isin(MAJOR), nodes["community_id"].astype(str), "minor")
nodes[["node_id", "tmdb_id", "community_id", "community_group", "within_community_strength", "participation_coefficient",
       "main_external_community", "external_share"]].to_csv(POSITIONS_FILE, index=False)
print(f"  ✓ processed/network/{POSITIONS_FILE.name}")
print(f"Edges inside a community: {e['internal'].mean():.1%} · "
      f"mean similarity inside {e.loc[e['internal'], 'w'].mean():.3f} vs between {e.loc[~e['internal'], 'w'].mean():.3f}")
display(cohesion.sort_values("cohesion_internal_share").head(8).round(3))


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 6 — Title families (display-only franchise cap) and franchise concentration
# ═══════════════════════════════════════════════════════════════════════════
def title_family(titles):
    """Vectorised version of v1's make_title_family (same output; verified on test titles)."""
    raw = titles.fillna("").astype(str).str.lower().str.strip()
    fam = pd.Series(np.nan, index=raw.index, dtype=object)
    for family, pattern in CFG["franchise_patterns"].items():        # first matching pattern wins, as in v1
        fam = fam.mask(fam.isna() & raw.str.contains(pattern, regex=True), family)
    t = (raw.str.replace(r"['’]s\b", "", regex=True)
            .str.replace(r"[^a-z0-9\s]", " ", regex=True)
            .str.replace(r"\b(the|a|an)\b", " ", regex=True)
            .str.replace(r"\b(part|chapter|episode|vol|volume|season|book|movie|film|return|revenge|rise|fall|curse|legend)\b\s*[ivx0-9]*", " ", regex=True)
            .str.replace(r"\b[ivx]{1,5}\b", " ", regex=True)
            .str.replace(r"\b\d{1,4}\b", " ", regex=True)
            .str.replace(r"\s+", " ", regex=True).str.strip())
    t = t.mask(t == "", raw)
    return fam.fillna(t)

nodes["title_family"] = title_family(nodes["title"])
nodes["visible"] = nodes["vote_count"].fillna(0) >= CFG["display_min_vote_count"]
print("Unique title families:", nodes["title_family"].nunique(), "of", len(nodes), "films")

franchise_concentration = (nodes.groupby("title_family")
    .agg(n_titles=("title", "count"),
         example_titles=("title", lambda x: "; ".join(x.dropna().astype(str).head(8))),
         communities=("community_id", lambda x: "; ".join(map(str, sorted(x.dropna().unique())[:8]))),
         n_communities=("community_id", "nunique"),
         median_vote_count=("vote_count", "median"), max_weighted_degree=("weighted_degree", "max"))
    .reset_index().sort_values(["n_titles", "max_weighted_degree"], ascending=[False, False]))
display(franchise_concentration.head(15))
save_v1(franchise_concentration, "franchise_concentration_diagnostic.csv")

fam_counts = nodes.groupby(["community_id", "title_family"]).size().rename("n").reset_index() \
                   .sort_values(["community_id", "n"], ascending=[True, False]).drop_duplicates("community_id")
family_dominance = pd.DataFrame({
    "community_id": fam_counts["community_id"].values,
    "largest_title_family": np.where(fam_counts["n"].values > 1, fam_counts["title_family"].values, ""),
    "largest_title_family_share": (fam_counts["n"].values / fam_counts["community_id"].map(
        nodes["community_id"].value_counts()).values).round(3)})
family_dominance.loc[family_dominance["largest_title_family"] == "", "largest_title_family_share"] = 0.0
multi = franchise_concentration[franchise_concentration["n_titles"] >= 3]
print(f"Title families with ≥ 3 films: {len(multi)}; "
      f"{(multi['n_communities'] == 1).mean():.0%} of them sit entirely in one community "
      "(franchises are pulled together by shared names and plots).")


## 3. The communities at a glance

Size, robustness (mean membership confidence of its films) and cohesion. Robust communities can be named with confidence; **tentative** ones (robustness < 0.5) are real regions of the network but with blurred borders, and the thesis should describe them as such.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 7 — Community size overview (v1 file + robustness and cohesion)
# ═══════════════════════════════════════════════════════════════════════════
community_size_table = (nodes.groupby("community_id")
    .agg(n_films=("node_id", "count"), median_year=("release_year", "median"),
         min_year=("release_year", "min"), max_year=("release_year", "max"),
         median_vote_count=("vote_count", "median"), median_vote_average=("vote_average", "median"),
         revenue_available_pct=("revenue_clean", lambda x: round(x.notna().mean() * 100, 2)),
         top_genre=("primary_genre", lambda x: x.value_counts().index[0] if x.notna().any() else np.nan))
    .reset_index().merge(cohesion, on="community_id", how="left")
    .sort_values("n_films", ascending=False).reset_index(drop=True))
community_size_table["share_of_network_pct"] = (community_size_table["n_films"] / len(nodes) * 100).round(2)
community_size_table = community_size_table.merge(family_dominance, on="community_id", how="left")
community_size_table["is_major"] = community_size_table["community_id"].isin(MAJOR)
if HAS_CONF:
    community_size_table["tentative"] = community_size_table["community_robustness"] < CFG["tentative_robustness"]
display(community_size_table.head(25).round(3))
save_v1(community_size_table, "community_size_overview.csv")


## 4. What the communities are made of

**Genre.** If communities were just genres, knowing a film's community would tell you its genre (NMI near 1). A low NMI with clearly themed communities means the plots group films *across* genre lines — the core claim of a plot-based approach.

**Words.** Class-based TF-IDF (Grootendorst, 2022) finds the words that are frequent in one community's overviews and rare elsewhere. Character names and places (spaCy named-entity recognition) are reported apart from theme words.

**Names as a confound.** Sentence embeddings react strongly to proper names: two unrelated films whose heroes are both called *Paul* can end up as neighbours. The cell measures how many edges join films that share a character name, against chance, and flags communities held together by one name.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 8 — Communities vs genres (NMI) and genre composition heatmap
# ═══════════════════════════════════════════════════════════════════════════
g = nodes.dropna(subset=["primary_genre"])
decade = (g["release_year"] // 10 * 10).astype(int)
genre_vs_comm = pd.DataFrame([
    {"comparison": "community vs primary genre", "NMI": normalized_mutual_info_score(g["primary_genre"], g["community_id"]),
     "ARI": adjusted_rand_score(g["primary_genre"], g["community_id"])},
    {"comparison": "community vs decade", "NMI": normalized_mutual_info_score(decade, g["community_id"]),
     "ARI": adjusted_rand_score(decade, g["community_id"])},
    {"comparison": "community vs shuffled genre (chance level)",
     "NMI": normalized_mutual_info_score(g["primary_genre"].sample(frac=1, random_state=42).values, g["community_id"]),
     "ARI": adjusted_rand_score(g["primary_genre"].sample(frac=1, random_state=42).values, g["community_id"])},
]).round(3)
display(genre_vs_comm)
save_new(genre_vs_comm, "communities_vs_genre_nmi.csv")

# per-community genre purity
ct = pd.crosstab(g["community_id"], g["primary_genre"])
p = ct.div(ct.sum(axis=1), axis=0)
purity = pd.DataFrame({"top_genre_share": p.max(axis=1),
                       "genre_entropy_norm": -(p * np.log(p.where(p > 0))).sum(axis=1) / np.log(p.shape[1])})
community_size_table = community_size_table.merge(purity.round(3), left_on="community_id", right_index=True, how="left")

# heatmap: major communities × most common genres (share of each community's films)
top_genres = g["primary_genre"].value_counts().head(12).index
H = p.loc[MAJOR, top_genres]
fig, ax = plt.subplots(figsize=(11, 0.34 * len(MAJOR) + 2))
im = ax.imshow(H.values, aspect="auto", cmap="Blues", vmin=0, vmax=max(0.5, H.values.max()))
ax.set_xticks(range(len(top_genres)), top_genres, rotation=40, ha="right")
ax.set_yticks(range(len(MAJOR)), [f"C{c} · {sizes[c]:,}" for c in MAJOR])
ax.grid(False)
for i in range(H.shape[0]):
    for j in range(H.shape[1]):
        v = H.values[i, j]
        if v >= 0.10:
            ax.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=7.5, color="white" if v > 0.35 else INK)
fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01, label="share of community's films")
ax.set_title("Genre mix of each narrative community")
subtitle(ax, f"Primary-genre mix of each major community · NMI(community, genre) = "
             f"{genre_vs_comm.loc[0, 'NMI']:.2f} (1 = communities are genres, 0 = unrelated)")
save(fig, H.reset_index().rename(columns={"index": "community_id"}), "fig_01_genre_composition")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 9 — Distinctive words, character names and places per community (class-based TF-IDF + spaCy NER)
# ═══════════════════════════════════════════════════════════════════════════
key_terms = pd.DataFrame({"community_id": sizes.index, "key_terms": "", "character_names": "", "places": ""})
name_edges = None
if CFG["key_terms"] and CORPUS_FILE.exists():
    from sklearn.feature_extraction.text import CountVectorizer
    text = nodes[["node_id", "tmdb_id", "community_id"]].merge(
        pd.read_csv(CORPUS_FILE, usecols=["tmdb_id", "overview"]), on="tmdb_id", how="left")
    docs = text["overview"].fillna("").astype(str)
    pattern = r"(?u)\b[^\W\d_]{3,}\b"
    cv = CountVectorizer(stop_words="english", token_pattern=pattern,
                         min_df=CFG["key_terms_min_df"], max_df=CFG["key_terms_max_df"])
    X = cv.fit_transform(docs)
    vocab = np.array(cv.get_feature_names_out())

    # capitalisation share of every word (a name is written with a capital almost every time)
    cv_raw = CountVectorizer(lowercase=False, token_pattern=pattern, min_df=2)
    R = np.asarray(cv_raw.fit_transform(docs).sum(axis=0)).ravel()
    raw_vocab = pd.Series(cv_raw.get_feature_names_out())
    caps = pd.DataFrame({"w": raw_vocab.str.lower(), "n": R, "cap": raw_vocab.str[0].str.isupper()})
    cap_share = caps.assign(nc=caps["n"] * caps["cap"]).groupby("w")[["nc", "n"]].sum()
    cap_share = pd.Series(vocab).map(cap_share["nc"] / cap_share["n"]).fillna(0).values
    capitalised = cap_share >= CFG["name_capital_share"]

    # people and places from spaCy NER (Colab ships spaCy; the small English model is fetched if missing)
    person_words, place_words, film_names = None, None, None
    if CFG["use_spacy_ner"]:
        try:
            import spacy
            try:
                nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer", "tagger", "attribute_ruler"])
            except OSError:
                from spacy.cli import download
                download("en_core_web_sm")
                nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer", "tagger", "attribute_ruler"])
            rows = []
            for i, d in enumerate(nlp.pipe(docs.tolist(), batch_size=256)):
                for ent in d.ents:
                    if ent.label_ in ("PERSON", "GPE", "LOC", "NORP"):
                        for w in re.findall(pattern, ent.text):
                            rows.append((i, w.lower(), "person" if ent.label_ == "PERSON" else "place"))
            ents = pd.DataFrame(rows, columns=["doc", "word", "kind"]).drop_duplicates()
            kind = ents.groupby(["word", "kind"]).size().unstack(fill_value=0)
            kind = kind.reindex(columns=["person", "place"], fill_value=0)
            person_words = set(kind.index[kind["person"] > kind["place"]])
            place_words = set(kind.index[kind["place"] >= kind["person"]])
            film_names = ents[ents["kind"] == "person"].assign(node_id=lambda d: text["node_id"].values[d["doc"]])
            print(f"spaCy NER: {len(person_words):,} name words, {len(place_words):,} place words.")
        except Exception as ex:
            print("spaCy NER unavailable, using the capitalisation rule only:", ex)
    if person_words is not None:
        is_name = capitalised & np.isin(vocab, list(person_words))
        is_place = capitalised & np.isin(vocab, list(place_words)) & ~is_name
    else:
        is_name, is_place = capitalised, np.zeros(len(vocab), dtype=bool)

    comm_codes, comm_index = pd.factorize(text["community_id"])
    M = sparse.csr_matrix((np.ones(len(docs)), (comm_codes, np.arange(len(docs)))), shape=(len(comm_index), len(docs)))
    C = (M @ X).toarray().astype(float)                                  # word counts per community
    tf = C / np.maximum(C.sum(axis=1, keepdims=True), 1)
    idf = np.log(1 + C.sum(axis=1).mean() / np.maximum(C.sum(axis=0), 1))
    ctfidf = tf * idf

    def top_words(mask, n):
        s_ = np.where(mask[None, :], ctfidf, -1)
        idx = np.argsort(-s_, axis=1)[:, :n]
        return ["; ".join(vocab[row][s_[i, row] > 0]) for i, row in enumerate(idx)]
    key_terms = pd.DataFrame({"community_id": comm_index,
                              "key_terms": top_words(~is_name & ~is_place, CFG["key_terms_n"]),
                              "character_names": top_words(is_name, 6),
                              "places": top_words(is_place, 5)})
    # how many of a community's overviews mention its most distinctive character name?
    if is_name.any():
        s_names = np.where(is_name[None, :], ctfidf, -1)
        top_idx = s_names.argmax(axis=1)
        present = (X[:, top_idx] > 0).toarray()                            # films × communities
        share = np.array([present[comm_codes == k, k].mean() for k in range(len(comm_index))])
        key_terms["top_name"] = np.where(s_names.max(axis=1) > 0, vocab[top_idx], "")
        key_terms["top_name_film_share"] = np.where(key_terms["top_name"] != "", share, 0).round(3)
    print(f"Vocabulary {len(vocab):,} words: {is_name.sum():,} names, {is_place.sum():,} places kept apart from themes.")
    display(key_terms.set_index("community_id").loc[MAJOR].head(33))

    # ── do films get linked because their overviews share a character name? ────
    if film_names is not None and len(film_names):
        fn = film_names[["node_id", "word"]].drop_duplicates()
        def share_with_name(a, b):
            pairs_ = pd.DataFrame({"a": a, "b": b, "k": np.arange(len(a))})
            hit = pairs_.merge(fn, left_on="a", right_on="node_id").merge(fn, left_on=["b", "word"],
                                                                          right_on=["node_id", "word"])
            return hit["k"].nunique() / len(a)
        obs = share_with_name(e["a"].values, e["b"].values)
        rng = np.random.default_rng(42)
        rand = share_with_name(e["a"].values, rng.permutation(e["b"].values))
        inside = share_with_name(e.loc[e["internal"], "a"].values, e.loc[e["internal"], "b"].values)
        name_edges = pd.DataFrame([
            {"measure": "edges whose two overviews share a character name", "value": round(obs, 4)},
            {"measure": "same, random film pairs (chance level)", "value": round(rand, 4)},
            {"measure": "ratio observed / chance", "value": round(obs / max(rand, 1e-9), 1)},
            {"measure": "same, edges inside a community", "value": round(inside, 4)},
        ])
        display(name_edges)
        save_new(name_edges, "edges_sharing_character_names.csv")
else:
    print("Key terms skipped (CFG['key_terms'] is False or df_network_corpus.csv is missing).")
for c, v in [("top_name", ""), ("top_name_film_share", 0.0)]:
    if c not in key_terms:
        key_terms[c] = v
save_new(key_terms, "community_key_terms.csv")

# ── franchise-driven vs character-name-driven communities ──────────────────
community_size_table = community_size_table.merge(key_terms[["community_id", "top_name", "top_name_film_share"]],
                                                  on="community_id", how="left")
community_size_table["franchise_driven"] = community_size_table["largest_title_family_share"] >= CFG["franchise_family_share"]
community_size_table["name_driven"] = community_size_table["top_name_film_share"].fillna(0) >= CFG["name_film_share"]
community_size_table["franchise_or_name_driven"] = community_size_table["franchise_driven"] | community_size_table["name_driven"]
fd = community_size_table[community_size_table["franchise_or_name_driven"]]
print(f"\nFranchise-driven: {int(community_size_table['franchise_driven'].sum())} communities · "
      f"character-name-driven: {int(community_size_table['name_driven'].sum())} "
      f"({int((community_size_table['name_driven'] & community_size_table['is_major']).sum())} major).")
print("A name-driven community groups films because their overviews mention the same first name "
      "(e.g. several unrelated films whose hero is called Paul). That is an artefact of the embedding, not a shared plot.")
display(fd[["community_id", "n_films", "is_major", "largest_title_family", "largest_title_family_share", "top_name",
            "top_name_film_share"]].head(30))
save_new(fd, "franchise_or_name_driven_communities.csv")


## 5. The narrative neighbourhood map

Which communities border which? Only the similarity that *leaves* each community is used. For every pair, the similarity flowing between them is compared with what it would be if each community's outside links went to other communities at random (**lift**). The map draws every major community as a circle (area = films) and links each one to its two strongest neighbours; communities that share many links sit close together.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 10 — Community adjacency (lift) + community map
# ═══════════════════════════════════════════════════════════════════════════
import networkx as nx
pair = pd.concat([e[["ca", "cb", "w"]], e[["cb", "ca", "w"]].set_axis(["ca", "cb", "w"], axis=1)])
W = pair.groupby(["ca", "cb"])["w"].sum().unstack(fill_value=0.0)
off_diag = ~np.eye(len(W), dtype=bool)
X_out = W.where(off_diag, 0.0)                # similarity leaving each community (internal similarity removed)
Xs = X_out.sum(axis=1)
lift = (X_out / np.outer(Xs, Xs) * Xs.sum()).where(off_diag)
ext_share = X_out.div(Xs, axis=0)

neighbours_long = (lift.stack().rename("lift").reset_index()
                   .rename(columns={"ca": "community_id", "cb": "neighbour_community"})
                   .merge(ext_share.stack().rename("share_of_external_similarity").reset_index()
                          .rename(columns={"ca": "community_id", "cb": "neighbour_community"}),
                          on=["community_id", "neighbour_community"])
                   .sort_values(["community_id", "share_of_external_similarity"], ascending=[True, False]))
save_new(neighbours_long.round(4), "community_neighbourhood_long.csv")
top_nb = (neighbours_long.groupby("community_id").head(3)
          .assign(s=lambda d: "C" + d["neighbour_community"].astype(str) + " ("
                  + (d["share_of_external_similarity"] * 100).round(0).astype(int).astype(str) + "%)")
          .groupby("community_id")["s"].agg("; ".join).rename("main_neighbour_communities"))
community_size_table = community_size_table.merge(top_nb, left_on="community_id", right_index=True, how="left")

# community map: major communities, each linked to its 2 strongest major neighbours (by lift)
nl = neighbours_long[neighbours_long["community_id"].isin(MAJOR) & neighbours_long["neighbour_community"].isin(MAJOR)]
links = nl.sort_values(["community_id", "lift"], ascending=[True, False]).groupby("community_id").head(2)
H = nx.Graph()
H.add_nodes_from(MAJOR)
for r in links.itertuples():
    H.add_edge(int(r.community_id), int(r.neighbour_community), lift=float(r.lift))
pos = nx.spring_layout(H, weight="lift", seed=42, k=1.6 / np.sqrt(len(MAJOR)), iterations=300)
fig, ax = plt.subplots(figsize=(11, 9))
for u, v, d in H.edges(data=True):
    ax.plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]], color=AXIS, linewidth=0.6 + 0.9 * min(d["lift"], 4), zorder=1)
xy = np.array([pos[c] for c in MAJOR])
area = 60 + 1400 * sizes[MAJOR].values / sizes[MAJOR].max()
tentative = (community_size_table.set_index("community_id").reindex(MAJOR)["tentative"].fillna(False).values
             if "tentative" in community_size_table else np.zeros(len(MAJOR), bool))
ax.scatter(xy[:, 0], xy[:, 1], s=area, color=np.where(tentative, SERIES[1], SERIES[0]), alpha=0.85,
           edgecolor=SURFACE, linewidth=1.2, zorder=2)
for c, (x_, y_) in zip(MAJOR, xy):
    ax.annotate(f"C{c}", (x_, y_), ha="center", va="center", fontsize=8, color="white", fontweight="bold", zorder=3)
ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
for sp in ax.spines.values():
    sp.set_visible(False)
ax.set_title("Map of the narrative communities")
subtitle(ax, "Each community is linked to its two closest neighbours (highest lift) · circle area = films · "
             "orange = tentative · labels: community_labels_v2_manual.csv")
save(fig, links.round(4), "fig_02_community_neighbourhood_map")


## 6. Labelling the communities

**How it works**
1. Notebook 03 v2 matched the new communities to the ones you labelled before and wrote `community_labels_draft_v2.csv`.
2. This cell creates **`outputs/tables/community_labels_v2_manual.csv`** (once). Carried labels are pre-filled; every row carries the evidence above (key words, names, core films, genres, neighbours, robustness).
3. **You edit only that file:** fill or correct `suggested_label_manual`, optionally a `short_label` (≤ 30 characters, for figures) and `interpretation_note_manual`, and write `yes` in `label_reviewed`. Rows marked *carried* are already reviewed.
4. Re-run this notebook (from Cell 1, or just Cells 11–17). When every major community has a reviewed label, the file the bricks read — `community_manual_labeling_template_labeled.csv` — is written, in its v1 columns.

Re-running never overwrites what you typed; evidence columns are refreshed. If Notebook 03 is re-run and the communities change, your file is kept as `…_stale_<date>.csv` and a fresh one is started from the new carry-over.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 11 — Labelling evidence per community (v1 helper columns + new evidence)
# ═══════════════════════════════════════════════════════════════════════════
def top_values_as_string(df, col, n):
    vc = df.groupby("community_id")[col].value_counts().groupby(level=0).head(n)
    return (vc.index.get_level_values(1).astype(str) + " (" + vc.astype(str).values + ")") \
             .to_series(index=vc.index.get_level_values(0)).groupby(level=0).agg("; ".join)

def titles_by(df, metric, n, ascending=False):
    """Top-n films per community by `metric`, one per title family, as 'Title (year)'."""
    t = (df.sort_values(["community_id", metric], ascending=[True, ascending])
           .drop_duplicates(["community_id", "title_family"]).groupby("community_id").head(n))
    lab = t["title"].astype(str) + " (" + t["release_year"].astype("Int64").astype(str) + ")"
    return lab.groupby(t["community_id"]).agg("; ".join)

CENTRAL = CFG["central_metric"]
nodes["_core_key"] = nodes["is_core"].astype(int) if CFG["core_first"] else 0
nodes["_central_key"] = nodes["_core_key"] * 1e6 + nodes[CENTRAL]       # core films first, then by centrality

helper = community_size_table[["community_id", "n_films", "share_of_network_pct"]].copy().set_index("community_id")
helper["top_primary_genres"] = top_values_as_string(nodes, "primary_genre", 6)
helper["median_year"] = community_size_table.set_index("community_id")["median_year"]
yr = nodes.groupby("community_id")["release_year"].agg(["min", "max"]).astype(int)
helper["year_range"] = yr["min"].astype(str) + "-" + yr["max"].astype(str)
for c in ["median_vote_count", "median_vote_average", "revenue_available_pct"]:
    helper[c] = community_size_table.set_index("community_id")[c]
helper["central_titles_weighted_degree"] = titles_by(nodes, "_central_key", 8)
helper["bridging_titles_betweenness"] = titles_by(nodes, BETW, 6)
helper["visible_titles_vote_count"] = titles_by(nodes, "vote_count", 8)
# new evidence
helper["core_exemplars_visible"] = titles_by(nodes[nodes["visible"]], "_central_key", 6)
helper["well_known_examples"] = titles_by(nodes[nodes["vote_count"] >= CFG["well_known_min_votes"]], "_central_key", 6)
helper = helper.join(key_terms.set_index("community_id").drop(columns=["top_name", "top_name_film_share"]))
helper["visible_titles_vote_count"] = helper["visible_titles_vote_count"]   # v1 column kept as is
extra_cols = ["community_robustness", "core_share", "cohesion_internal_share", "top_genre_share",
              "main_neighbour_communities", "tentative", "largest_title_family", "largest_title_family_share",
              "top_name", "top_name_film_share", "franchise_driven", "name_driven"]
helper = helper.join(community_size_table.set_index("community_id")[[c for c in extra_cols if c in community_size_table]])
helper = helper.reset_index().sort_values("n_films", ascending=False).reset_index(drop=True)
helper.insert(0, "interpretation_priority", range(1, len(helper) + 1))
print("Columns:", list(helper.columns))
display(helper.head(10))
save_v1(helper.assign(suggested_label_manual="", interpretation_note_manual=""), "community_label_helper_v2_manual.csv")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 12 — Your labels file: create once, refresh evidence, keep what you typed
# ═══════════════════════════════════════════════════════════════════════════
USER_COLS = ["suggested_label_manual", "short_label", "interpretation_note_manual", "label_reviewed"]
EVIDENCE_COLS = ["n_films", "community_robustness", "tentative", "name_driven", "franchise_driven", "key_terms",
                 "character_names", "places", "well_known_examples", "core_exemplars_visible", "top_primary_genres",
                 "main_neighbour_communities"]

# starting point: NB03's carry-over draft (if any)
start = pd.DataFrame({"community_id": helper["community_id"]})
if DRAFT_FILE.exists():
    draft = pd.read_csv(DRAFT_FILE)
    keep = [c for c in ["community_id", "carryover_status", "matched_old_community", "jaccard",
                        "suggested_label_manual", "interpretation_note_manual"] if c in draft]
    start = start.merge(draft[keep].rename(columns={"suggested_label_manual": "carried_label"}), on="community_id", how="left")
    start["suggested_label_manual"] = start["carried_label"]
    start["label_reviewed"] = np.where(start["carryover_status"].eq("carried") & ~is_blank(start["carried_label"]), "yes", "")
    print("Carry-over draft found:", start["carryover_status"].value_counts().to_dict())
else:
    print("No carry-over draft (community_labels_draft_v2.csv) — every community starts unlabelled.")
    for c in ["carryover_status", "matched_old_community", "jaccard", "carried_label"]:
        start[c] = np.nan
for c in USER_COLS:
    if c not in start:
        start[c] = ""

fp_of = lambda df: df.get("partition_fingerprint", pd.Series([None])).iloc[0] if len(df) else None
existing = pd.read_csv(MANUAL_FILE) if MANUAL_FILE.exists() else None
if existing is None or fp_of(existing) != PARTITION_FP:      # an earlier file made for exactly these communities?
    for old_file in sorted(MANUAL_FILE.parent.glob(f"{MANUAL_FILE.stem}_stale_*.csv"), reverse=True):
        if fp_of(pd.read_csv(old_file)) == PARTITION_FP:
            if existing is not None:
                MANUAL_FILE.rename(MANUAL_FILE.with_name(f"{MANUAL_FILE.stem}_stale_{datetime.now():%Y%m%d_%H%M%S}.csv"))
            existing = pd.read_csv(old_file)
            print(f"Restored your labels from {old_file.name} (made for these same communities).")
            break
if existing is not None and fp_of(existing) != PARTITION_FP:
    stale = MANUAL_FILE.with_name(f"{MANUAL_FILE.stem}_stale_{datetime.now():%Y%m%d_%H%M%S}.csv")
    MANUAL_FILE.rename(stale)
    print(f"⚠ Your labels file belongs to a different set of communities (NB03 was re-run). "
          f"Kept as {stale.name}; starting a fresh one.")
    existing = None
if existing is not None:                      # keep what you typed, take everything else fresh
    typed = existing.set_index("community_id")[[c for c in USER_COLS if c in existing]]
    start = start.set_index("community_id")
    for c in typed.columns:
        start[c] = typed[c].reindex(start.index).where(~is_blank(typed[c].reindex(start.index)), start[c])
    start = start.reset_index()
    print("Existing labels file found — your entries are kept, evidence refreshed.")

labels = (start.merge(helper[["community_id"] + [c for c in EVIDENCE_COLS if c in helper]], on="community_id", how="left")
               .sort_values("n_films", ascending=False))
labels["is_major"] = labels["community_id"].isin(MAJOR)
labels["partition_fingerprint"] = PARTITION_FP
for c in USER_COLS + ["carried_label"]:
    labels[c] = labels[c].where(~is_blank(labels[c]), "")
labels["label_reviewed"] = labels["label_reviewed"].str.strip().str.lower()
norm_label = labels["suggested_label_manual"].str.strip().str.lower()
duplicate = labels["is_major"] & ~is_blank(labels["suggested_label_manual"]) & \
            norm_label.where(labels["is_major"]).duplicated(keep=False) & labels["is_major"]
labels["status"] = np.select(
    [is_blank(labels["suggested_label_manual"]), duplicate, labels["label_reviewed"].eq("yes")],
    ["needs label", "duplicate label", "ready"], default="needs review")
if duplicate.any():
    print("⚠ Several major communities carry the same label (an old community split in two). Give each its own label:",
          labels.loc[duplicate, "community_id"].tolist())
order_cols = (["community_id", "is_major", "status"] + USER_COLS[:1] + ["short_label", "label_reviewed",
              "interpretation_note_manual", "carryover_status", "matched_old_community", "jaccard", "carried_label"]
              + [c for c in EVIDENCE_COLS if c in labels] + ["partition_fingerprint"])
labels = labels[order_cols]
labels.to_csv(MANUAL_FILE, index=False)
print(f"  ✓ tables/{MANUAL_FILE.name}  ← edit this file")

major_status = labels[labels["is_major"]]["status"].value_counts()
print("\nMajor communities:", major_status.to_dict())
display(labels[labels["is_major"] & labels["status"].ne("ready")]
        [["community_id", "n_films", "status", "carryover_status", "carried_label", "key_terms",
          "well_known_examples"]].head(40))


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 13 — Template (v1 file) and the labelled file the bricks read
# ═══════════════════════════════════════════════════════════════════════════
V1_TEMPLATE_COLS = ["interpretation_priority", "community_id", "n_films", "share_of_network_pct", "top_primary_genres",
                    "year_range", "median_year", "median_vote_count", "median_vote_average", "revenue_available_pct",
                    "central_titles_weighted_degree", "bridging_titles_betweenness", "visible_titles_vote_count",
                    "suggested_label_manual", "interpretation_note_manual"]
NEW_TEMPLATE_COLS = ["short_label", "status", "carryover_status", "community_robustness", "tentative",
                     "name_driven", "franchise_driven", "key_terms", "character_names", "places",
                     "well_known_examples", "core_exemplars_visible", "main_neighbour_communities"]
template = (helper[helper["community_id"].isin(MAJOR)]
            .merge(labels[["community_id", "suggested_label_manual", "interpretation_note_manual", "short_label",
                           "status", "carryover_status"]], on="community_id", how="left"))
template["interpretation_priority"] = range(1, len(template) + 1)
template = template[V1_TEMPLATE_COLS + [c for c in NEW_TEMPLATE_COLS if c in template]]
save_v1(template, "community_manual_labeling_template.csv")
save_v1(template[["community_id", "n_films", "share_of_network_pct", "top_primary_genres",
                  "central_titles_weighted_degree", "visible_titles_vote_count", "suggested_label_manual"]
                 + [c for c in ["community_robustness", "key_terms"] if c in template]],
        "major_community_thesis_table.csv")

ready = template["status"].eq("ready") if CFG["require_review_before_export"] else ~is_blank(template["suggested_label_manual"])
LABELS_COMPLETE = bool(ready.all())
if LABELS_COMPLETE:
    template.to_csv(LABELED_FILE, index=False)
    print(f"  ✓ tables/{LABELED_FILE.name} — all {len(template)} major communities labelled; the bricks can run.")
else:
    print(f"✗ {LABELED_FILE.name} NOT written: {int((~ready).sum())} of {len(template)} major communities still need a "
          f"{'reviewed ' if CFG['require_review_before_export'] else ''}label → edit {MANUAL_FILE.name}, then re-run.")
    if LABELED_FILE.exists():
        old = pd.read_csv(LABELED_FILE)
        same = (set(zip(old["community_id"], old["n_films"])) == set(zip(template["community_id"], template["n_films"])))
        if not same:
            print(f"⚠ The existing {LABELED_FILE.name} was made for the OLD network (different communities). "
                  "Do not run Notebook 05 or the bricks until labelling is finished — they would attach old labels "
                  "to new community ids.")

# label used in every table and figure below: your short label, else your label, else a placeholder
lab = labels.set_index("community_id")
display_label = lab["short_label"].where(~is_blank(lab["short_label"]), lab["suggested_label_manual"].map(lambda x: short(x) if x else ""))
display_label = display_label.where(~is_blank(display_label), "unlabelled")
nodes["community_label"] = nodes["community_id"].map(lab["suggested_label_manual"]).fillna("")
CLAB = {c: f"C{c} · {display_label.get(c, 'unlabelled')}" for c in sizes.index}


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 14 — Figure: community size and robustness (replaces v1 top_20_communities_by_size.png)
# ═══════════════════════════════════════════════════════════════════════════
t = community_size_table[community_size_table["is_major"]].copy()
t["label"] = t["community_id"].map(CLAB)
fig, ax = plt.subplots(figsize=(10, 0.33 * len(t) + 1.8))
y = np.arange(len(t))[::-1]
if HAS_CONF:
    colors = np.where(t["tentative"], SERIES[1], SERIES[0])
    ax.barh(y, t["n_films"], color=colors, height=0.62)
    for yi, (n, r) in zip(y, t[["n_films", "community_robustness"]].values):
        ax.text(n, yi, f"  {int(n):,} · robustness {r:.2f}", va="center", fontsize=8, color=INK_2)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color=SERIES[0], label=f"robust (≥ {CFG['tentative_robustness']})"),
                       Patch(color=SERIES[1], label="tentative — describe with care")],
              loc="upper center", bbox_to_anchor=(0.5, -0.09), ncol=2)
else:
    ax.barh(y, t["n_films"], color=SERIES[0], height=0.62)
ax.set_yticks(y, t["label"])
ax.set_xlabel("films")
ax.set_xlim(0, t["n_films"].max() * 1.35)
ax.grid(axis="y", visible=False)
ax.set_title("The narrative communities of the network")
subtitle(ax, f"{len(t)} major communities (≥ {CFG['min_community_size_for_interpretation']} films) · "
             f"{t['n_films'].sum() / len(nodes):.0%} of the {len(nodes):,} films")
save(fig, t.drop(columns=["label"]), "fig_03_community_size_robustness", also_v1="top_20_communities_by_size.png")


## 7. Central films

Global tables keep v1's definitions (weighted degree, betweenness, PageRank) so the bricks keep working; betweenness is now exact. Per community, representative films are ranked by **within-community strength**, core films first (`CFG['central_metric']`). All example tables apply the display rules: at least 50 votes and one film per title family (the network itself is never changed).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 15 — Global centrality tables (v1 files, new columns at the end)
# ═══════════════════════════════════════════════════════════════════════════
V1_COLS = ["title", "release_year", "primary_genre", "vote_count", "vote_average", "community_id", "degree",
           "weighted_degree", "pagerank", "betweenness_centrality_sampled", "eigenvector_centrality", "clustering_coefficient"]
NEW_COLS = ["betweenness_centrality", "membership_confidence", "within_community_strength",
            "participation_coefficient", "community_label", "title_family"]
def compact(df):
    return df[[c for c in V1_COLS + NEW_COLS if c in df.columns]].copy()

def top_unique(df, metric, n):
    return df.sort_values(metric, ascending=False).drop_duplicates("title_family").head(n)

N = CFG["top_n_global"]
vis = nodes[nodes["visible"]]
tables = {
    "top_weighted_degree_raw.csv":                      compact(nodes.sort_values("weighted_degree", ascending=False).head(N)),
    "top_weighted_degree_franchise_capped.csv":         compact(top_unique(nodes, "weighted_degree", N)),
    "top_betweenness_franchise_capped.csv":             compact(top_unique(nodes, BETW, N)),
    "top_pagerank_franchise_capped.csv":                compact(top_unique(nodes, "pagerank", N)),
    "top_weighted_degree_visible_franchise_capped.csv": compact(top_unique(vis, "weighted_degree", N)),
    "top_betweenness_visible_franchise_capped.csv":     compact(top_unique(vis, BETW, N)),
    "top_pagerank_visible_franchise_capped.csv":        compact(top_unique(vis, "pagerank", N)),
}
for name, t in tables.items():
    save_v1(t, name)
raw = tables["top_weighted_degree_raw.csv"]
print(f"\nFranchise effect: the raw top-{N} by weighted degree holds {raw['title_family'].nunique()} distinct title families "
      f"({N - raw['title_family'].nunique()} rows are repeats of the same franchise).")
display(tables["top_weighted_degree_visible_franchise_capped.csv"].head(15))


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 16 — Representative films per community (v1 files; one sort + groupby, no loops)
# ═══════════════════════════════════════════════════════════════════════════
def per_community(df, key, n):
    t = (df.sort_values(["community_id", key], ascending=[True, False])
           .drop_duplicates(["community_id", "title_family"]).groupby("community_id").head(n).copy())
    t["rank"] = t.groupby("community_id").cumcount() + 1
    return t

KEEP = ["community_id", "title", "release_year", "primary_genre", "vote_count", "vote_average", "weighted_degree",
        "betweenness_centrality_sampled", "pagerank"]
KEEP_NEW = ["within_community_strength", "membership_confidence", "is_core", "participation_coefficient", "community_label"]
KEEP_NEW = [c for c in KEEP_NEW if c in nodes]

top_central = per_community(vis, "_central_key", CFG["top_n_per_community_display"]) \
    .rename(columns={"rank": "rank_within_community"})
top_central = (top_central[["community_id", "rank_within_community"] + KEEP[1:] + KEEP_NEW]
               .merge(community_size_table[["community_id", "n_films", "top_genre"]], on="community_id", how="left")
               .sort_values(["n_films", "community_id", "rank_within_community"], ascending=[False, True, True]))
top_central = top_central[[c for c in top_central if c not in KEEP_NEW] + KEEP_NEW]   # v1 order, new columns last
save_v1(top_central, "top_visible_central_films_per_community.csv")

n = CFG["top_n_per_community"]
major_nodes = nodes[nodes["community_id"].isin(MAJOR)]
reps = pd.concat([
    per_community(major_nodes, "_central_key", n).assign(selection_type="central_within_community"),
    per_community(major_nodes, BETW, n).assign(selection_type="bridging_betweenness"),
    per_community(major_nodes, "vote_count", n).assign(selection_type="visible_examples"),
], ignore_index=True)
reps = reps[["community_id", "selection_type", "rank"] + KEEP[1:] + KEEP_NEW]
save_v1(reps, "community_representative_films_long.csv")

show = top_central[top_central["community_id"].isin(MAJOR[:8])]
display(show[["community_id", "rank_within_community", "title", "release_year", "vote_count"]
             + [c for c in ["membership_confidence", "community_label"] if c in show]])


## 8. Bridge films

Exact betweenness ranks films that lie on many shortest paths between other films. The **participation coefficient** says whether that is because the film's own neighbours are spread across communities (a true crossover plot) or because it links two otherwise separate areas through a few edges. Each bridge is listed with the two communities it connects.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 17 — Bridging films (v1 file + which communities each bridge joins) and a figure
# ═══════════════════════════════════════════════════════════════════════════
b = top_unique(nodes, BETW, CFG["top_n_bridges"])
bridges = compact(b).join(b[["main_external_community", "external_share"]])
full = {c: f"C{c} · {lab['suggested_label_manual'].get(c, '') or 'unlabelled'}" for c in sizes.index}
bridges["bridges_between"] = (bridges["community_id"].map(full) + "  ↔  "
                              + bridges["main_external_community"].map(lambda c: full.get(c, "—") if pd.notna(c) else "—"))
save_v1(bridges, "bridging_films_betweenness.csv")
display(bridges[["title", "release_year", "vote_count", BETW, "participation_coefficient", "bridges_between"]].head(15))

bp = bridges.dropna(subset=["main_external_community"])
lo = np.minimum(bp["community_id"], bp["main_external_community"].astype(int))
hi = np.maximum(bp["community_id"], bp["main_external_community"].astype(int))
pairs = ("C" + lo.astype(str) + " ↔ C" + hi.astype(str)).value_counts().head(10)
print("Community pairs joined by the most top bridges:"); display(pairs.to_frame("bridges"))

rho = nodes[["participation_coefficient", BETW]].corr(method="spearman").iloc[0, 1]
# figure: the best-known bridge films (≥ display_min_vote_count votes), with the two communities they join
tb = (vis.dropna(subset=["main_external_community"]).sort_values(BETW, ascending=False)
         .drop_duplicates("title_family").head(15).iloc[::-1])
fig, ax = plt.subplots(figsize=(10, 0.42 * len(tb) + 1.6))
y = np.arange(len(tb))
ax.hlines(y, 0, tb[BETW] * 1e3, color=AXIS, linewidth=1.5)
ax.scatter(tb[BETW] * 1e3, y, s=45, color=SERIES[0], zorder=3)
for yi, r in zip(y, tb.itertuples()):
    ax.annotate(f"C{r.community_id} ↔ C{int(r.main_external_community)} · P = {r.participation_coefficient:.2f}",
                (getattr(r, BETW) * 1e3, yi), xytext=(8, 0), textcoords="offset points", va="center",
                fontsize=8, color=INK_2)
ax.set_yticks(y, [f"{short(t, 34)} ({int(yr)})" for t, yr in zip(tb["title"], tb["release_year"])])
ax.set_xlim(0, tb[BETW].max() * 1e3 * 1.45)
ax.set_xlabel("betweenness × 1,000 (exact)" if BETW == "betweenness_centrality" else "betweenness × 1,000 (sampled)")
ax.grid(axis="y", visible=False)
ax.set_title("Best-known bridge films")
subtitle(ax, f"Films with ≥ {CFG['display_min_vote_count']} votes, one per title family · P = participation coefficient · "
             f"across all films, Spearman ρ(P, betweenness) = {rho:.2f}")
save(fig, tb[["title", "release_year", "community_id", "main_external_community", "participation_coefficient", BETW]],
     "fig_04_bridges_participation")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 18 — Peripheral films (v1 file + confidence) and boundary films
# ═══════════════════════════════════════════════════════════════════════════
PCOLS = ["title", "release_year", "primary_genre", "vote_count", "vote_average", "community_id", "degree",
         "weighted_degree", "clustering_coefficient"] + [c for c in ["membership_confidence", "community_label"] if c in nodes]
peripheral = (vis.sort_values(["degree", "weighted_degree", "clustering_coefficient"])[PCOLS].head(CFG["n_peripheral"]))
save_v1(peripheral, "peripheral_films_candidates.csv")
print("Peripheral = few, weak plot neighbours. Not 'bad' films — semantically unusual ones.")
display(peripheral.head(10))

if HAS_CONF:
    print(f"Boundary films (in their community in < {CFG['boundary_confidence']:.0%} of single runs): "
          f"{int(nodes['is_boundary'].sum()):,} ({nodes['is_boundary'].mean():.1%})")
    boundary = (vis[vis["community_id"].isin(MAJOR)].sort_values("membership_confidence")
                [["title", "release_year", "vote_count", "community_id", "membership_confidence",
                  "main_external_community", "participation_coefficient", "community_label"]].head(50))
    save_new(boundary, "boundary_films_low_confidence.csv")
    print("Well-known boundary films (communities disagree across runs):"); display(boundary.head(10))


## 8b. Minor communities

Communities below the size threshold are not labelled one by one. They are described together here, so the thesis can say what they are (many are franchise or character clusters), and Notebook 05 can pool them into a single **minor** group (`community_group` in `processed/network/node_positions_nb04.csv`) instead of fitting dozens of tiny dummies.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 18b — Minor communities summary
# ═══════════════════════════════════════════════════════════════════════════
minor = community_size_table[~community_size_table["is_major"]].copy()
if len(minor):
    minor["visible_examples"] = minor["community_id"].map(titles_by(nodes, "vote_count", 5))
    minor["key_terms"] = minor["community_id"].map(key_terms.set_index("community_id")["key_terms"])
    cols = ["community_id", "n_films", "top_genre", "visible_examples", "key_terms", "largest_title_family",
            "largest_title_family_share", "top_name", "top_name_film_share", "franchise_or_name_driven",
            "main_neighbour_communities"] + [c for c in ["community_robustness"] if c in minor]
    minor = minor[cols]
    print(f"{len(minor)} minor communities hold {minor['n_films'].sum():,} films ({minor['n_films'].sum() / len(nodes):.1%}); "
          f"{int(minor['franchise_or_name_driven'].sum())} of them are franchise/name-driven.")
    display(minor.head(20))
    save_new(minor, "minor_communities.csv")
else:
    print("No minor communities.")


## 9. Reception by community (descriptive only)

A first look at whether communities differ in rating and revenue. This is **not** the success model — that is Notebook 05, with controls for year, genre and visibility. The Kruskal–Wallis ε² effect size says how much of the variation lies between communities (0 = none, 1 = all).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 19 — Community outcome summary (v1 file) + effect sizes + figure
# ═══════════════════════════════════════════════════════════════════════════
community_outcome_summary = (nodes.groupby("community_id")
    .agg(n_films=("node_id", "count"),
         revenue_available_pct=("revenue_clean", lambda x: round(x.notna().mean() * 100, 2)),
         median_revenue=("revenue_clean", "median"), mean_log_revenue=("log_revenue", "mean"),
         median_vote_average=("vote_average", "median"), median_vote_count=("vote_count", "median"),
         median_popularity=("popularity", "median"), median_weighted_degree=("weighted_degree", "median"),
         median_betweenness=(BETW, "median"))
    .reset_index().sort_values("n_films", ascending=False))
community_outcome_summary["community_label"] = community_outcome_summary["community_id"].map(lab["suggested_label_manual"])
if HAS_CONF:
    community_outcome_summary["community_robustness"] = community_outcome_summary["community_id"].map(
        community_size_table.set_index("community_id")["community_robustness"])
save_v1(community_outcome_summary, "community_outcome_summary_descriptive.csv")

def kw(col):
    d = nodes[nodes["community_id"].isin(MAJOR)].dropna(subset=[col])
    groups = [x[col].values for _, x in d.groupby("community_id")]
    H, pval = kruskal(*groups)
    return {"outcome": col, "films": len(d), "H": round(H, 1), "p_value": pval,
            "epsilon_squared": round((H - len(groups) + 1) / (len(d) - len(groups)), 3)}
effects = pd.DataFrame([kw(c) for c in ["vote_average", "log_revenue", "popularity"] if c in nodes])
display(effects)
save_new(effects, "community_outcome_effect_sizes.csv")

d = nodes[nodes["community_id"].isin(MAJOR)]
q = d.groupby("community_id").agg(r50=("vote_average", "median"), r25=("vote_average", lambda x: x.quantile(.25)),
                                   r75=("vote_average", lambda x: x.quantile(.75)), v50=("log_revenue", "median"),
                                   v25=("log_revenue", lambda x: x.quantile(.25)), v75=("log_revenue", lambda x: x.quantile(.75)),
                                   rev_n=("log_revenue", "count")).sort_values("r50")
fig, axes = plt.subplots(1, 2, figsize=(12, 0.32 * len(q) + 2), sharey=True)
y = np.arange(len(q))
for ax, (m, lo, hi, color, xl) in zip(axes, [("r50", "r25", "r75", SERIES[0], "TMDB rating"),
                                              ("v50", "v25", "v75", SERIES[2], "log revenue (films with revenue)")]):
    ax.hlines(y, q[lo], q[hi], color=color, alpha=0.35, linewidth=5)
    ax.plot(q[m], y, "o", color=color)
    ax.set_xlabel(xl)
    ax.grid(axis="y", visible=False)
axes[0].set_yticks(y, [CLAB[c] for c in q.index])
eps = effects.set_index("outcome")["epsilon_squared"]
axes[0].set_title("Rating and revenue by narrative community")
subtitle(axes[0], f"Median and interquartile range · ε² rating = {eps.get('vote_average', np.nan):.3f}, "
                  f"ε² log revenue = {eps.get('log_revenue', np.nan):.3f} · descriptive, no controls")
save(fig, q.reset_index(), "fig_05_reception_by_community")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 20 — Interpretation notes (written from the numbers; claims appear only if the data supports them)
# ═══════════════════════════════════════════════════════════════════════════
nmi = genre_vs_comm.loc[0, "NMI"]
lines = [
    "Network interpretation notes",
    "",
    f"The semantic network contains {len(nodes):,} films in its largest connected component and {len(edges):,} "
    "mutual-kNN similarity edges. Edges represent similarity between plot overviews, not success, fame or quality.",
    f"Community detection ({audit.get('community_method', 'see network_audit.csv')}) finds {nodes['community_id'].nunique()} "
    f"communities; {len(MAJOR)} have at least {CFG['min_community_size_for_interpretation']} films and cover "
    f"{sizes[MAJOR].sum() / len(nodes):.1%} of the network. {e['internal'].mean():.1%} of edges lie inside a community.",
]
if HAS_CONF:
    tent = community_size_table[community_size_table["is_major"] & community_size_table["tentative"]]
    lines.append(f"Membership confidence (share of single Louvain runs that keep a film with its consensus community): "
                 f"median {nodes['membership_confidence'].median():.2f}; {nodes['is_boundary'].mean():.1%} of films are "
                 f"boundary films (< {CFG['boundary_confidence']}). {nodes['is_core'].mean():.0%} of films are core films "
                 f"(≥ {CFG['core_internal_share']:.0%} of their similarity inside their community). "
                 + (f"{len(tent)} major {'community is' if len(tent) == 1 else 'communities are'} tentative (robustness < {CFG['tentative_robustness']}): "
                    f"{', '.join('C' + tent['community_id'].astype(str))}; describe {'its' if len(tent) == 1 else 'their'} boundaries as blurred."
                    if len(tent) else "All major communities are robust."))
lines.append(f"NMI between community and primary genre is {nmi:.2f} (chance level {genre_vs_comm.loc[2, 'NMI']:.2f}). "
             + ("Communities overlap little with primary-genre labels (part of this gap reflects that a single primary "
                "genre is a coarse label); several communities are thematic (crime, war, sci-fi, school) across genres."
                if nmi < 0.25 else
                "Communities partly follow genre lines but are not reducible to them."
                if nmi < 0.5 else "Communities follow genre lines closely; the plot-based grouping largely restates genre."))
cst = community_size_table
lines.append(f"{len(MAJOR)} communities are labelled; the other {len(sizes) - len(MAJOR)} minor communities hold "
             f"{sizes.drop(MAJOR).sum() / len(nodes):.1%} of films and are pooled as 'minor'. "
             f"{int(cst['franchise_driven'].sum())} communities are franchise-driven and {int(cst['name_driven'].sum())} "
             f"character-name-driven ({int((cst['name_driven'] & cst['is_major']).sum())} of them major).")
if name_edges is not None:
    nv = name_edges.set_index("measure")["value"]
    lines.append(f"{nv.iloc[0]:.1%} of edges join two films whose overviews share a character name "
                 f"({nv.iloc[2]:.0f}× the chance level of {nv.iloc[1]:.2%}). "
                 + ("Proper names therefore pull films together independently of plot; masking names before embedding "
                    "(Notebook 03) is recommended." if nv.iloc[2] >= 3 else "Names play only a minor role in the edges."))
lines.append("Weighted degree is bounded by k = 20 neighbours, so the most 'central' films are those whose overviews "
             "are most generic (similar to many others); they are typical, not influential, films.")
lines.append(f"Betweenness is {'exact' if BETW == 'betweenness_centrality' else 'sampled (approximate)'}. "
             f"Spearman correlation between participation coefficient and betweenness: {rho:.2f}.")
lines.append(f"The raw top-{N} by weighted degree contains {raw['title_family'].nunique()} distinct title families; "
             "thesis tables therefore apply a display-only franchise cap and a 50-vote visibility rule.")
if len(effects):
    lines.append("Descriptive differences in reception between major communities (Kruskal–Wallis ε²): "
                 + ", ".join(f"{r.outcome} {r.epsilon_squared:.3f}" for r in effects.itertuples())
                 + ". Formal modelling with controls follows in Notebook 05.")
lines.append(f"Labels: {int(major_status.get('ready', 0))} of {len(MAJOR)} major communities have reviewed labels"
             + ("." if LABELS_COMPLETE else f"; the labelled file for the bricks is not yet written."))
notes = "\n".join(lines)
(TABLES_DIR / "network_interpretation_notes.txt").write_text(notes, encoding="utf-8")
print(notes)
print("\n  ✓ tables/network_interpretation_notes.txt")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 21 — Output check
# ═══════════════════════════════════════════════════════════════════════════
brick_inputs = ["top_visible_central_films_per_community.csv", "top_weighted_degree_raw.csv",
                "top_weighted_degree_franchise_capped.csv", "top_weighted_degree_visible_franchise_capped.csv",
                "top_betweenness_visible_franchise_capped.csv", "bridging_films_betweenness.csv",
                LABELED_FILE.name]
other_v1 = ["network_interpretation_basic_checks.csv", "remaining_event_like_titles_check.csv",
            "community_size_overview.csv", "franchise_concentration_diagnostic.csv",
            "top_betweenness_franchise_capped.csv", "top_pagerank_franchise_capped.csv",
            "top_pagerank_visible_franchise_capped.csv", "community_representative_films_long.csv",
            "community_label_helper_v2_manual.csv", "community_manual_labeling_template.csv",
            "major_community_thesis_table.csv", "peripheral_films_candidates.csv",
            "community_outcome_summary_descriptive.csv", "network_interpretation_notes.txt", MANUAL_FILE.name]
rows = [{"file": f, "group": "read by bricks", "exists": (TABLES_DIR / f).exists()} for f in brick_inputs]
rows += [{"file": f, "group": "v1 table", "exists": (TABLES_DIR / f).exists()} for f in other_v1]
rows += [{"file": f"interpretation/{p.name}", "group": "new", "exists": True} for p in sorted(INT_TABLES.glob("*.csv"))]
rows += [{"file": f"figures/interpretation/{p.name}", "group": "new figure", "exists": True} for p in sorted(INT_FIGS.glob("*.png"))]
out = pd.DataFrame(rows)
display(out)
if not LABELS_COMPLETE:
    print(f"Next: open {MANUAL_FILE.name}, label the rows marked 'needs label' / 'needs review', set label_reviewed = yes, re-run.")


## What comes next

* **Finish the labels** in `community_labels_v2_manual.csv` until Cell 13 reports that the labelled file was written. Use `key_terms`, `core_exemplars_visible` and `carried_label` together; for *tentative* communities choose a broad label.
* Then **Notebook 05 v2** (success and star power): it reads the labels and the new columns and `membership_confidence`, exact betweenness and — from `processed/network/node_positions_nb04.csv` — the participation coefficient, if wanted as a regressor.
* For the Results chapter: Figure 1 (genre composition + NMI) answers "are these just genres?", Figure 2 is the map of narrative neighbourhoods, Figure 4 describes bridge films.